In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

In [3]:
cd C:\Deadpool\Github_files\F1-ENE-datapipeline-project

C:\Deadpool\Github_files\F1-ENE-datapipeline-project


In [4]:
BASE = Path("data/bronze/weather/season=2024")

paths = {
    "r1":  BASE / "round=1"  / "session=R" / "data.parquet",
    "rmid": BASE / "round=15" / "session=R" / "data.parquet",
    "rlast": BASE / "round=22" / "session=R" / "data.parquet",
}

for k, p in paths.items():
    print(k, "->", p, "| exists:", p.exists())


r1 -> data\bronze\weather\season=2024\round=1\session=R\data.parquet | exists: True
rmid -> data\bronze\weather\season=2024\round=15\session=R\data.parquet | exists: True
rlast -> data\bronze\weather\season=2024\round=22\session=R\data.parquet | exists: True


In [5]:
df1 = pd.read_parquet(paths["r1"], engine="pyarrow")
df1.shape

(157, 12)

In [6]:
df1.head(5)

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,season,event_round,session_type,ingestion_ts
0,0 days 00:00:14.093000,18.9,46.0,1017.1,False,26.5,162,0.9,2024,1,R,2025-12-15 23:39:57.830341
1,0 days 00:01:14.084000,18.9,46.0,1017.0,False,26.5,55,1.0,2024,1,R,2025-12-15 23:39:57.830341
2,0 days 00:02:14.093000,18.9,46.0,1017.0,False,26.5,55,1.0,2024,1,R,2025-12-15 23:39:57.830341
3,0 days 00:03:14.090000,18.9,45.0,1017.0,False,26.2,85,1.1,2024,1,R,2025-12-15 23:39:57.830341
4,0 days 00:04:14.091000,18.9,46.0,1017.0,False,26.2,178,1.0,2024,1,R,2025-12-15 23:39:57.830341


In [7]:
df1.dtypes

Time             timedelta64[ns]
AirTemp                  float64
Humidity                 float64
Pressure                 float64
Rainfall                    bool
TrackTemp                float64
WindDirection              int64
WindSpeed                float64
season                     int64
event_round                int64
session_type              object
ingestion_ts      datetime64[us]
dtype: object

In [10]:
timedelta_cols = df1.select_dtypes(include=["timedelta64[ns]"]).columns.tolist()
datetime_cols  = df1.select_dtypes(include=["datetime64[ns]"]).columns.tolist()
object_cols    = df1.select_dtypes(include=["object"]).columns.tolist()

print("timedelta cols:", timedelta_cols)
print("datetime cols:", datetime_cols)
print("object cols:", object_cols)
print("object cols count:", len(object_cols))


timedelta cols: ['Time']
datetime cols: ['ingestion_ts']
object cols: ['session_type']
object cols count: 1


In [11]:
null_rate = (df1.isna().mean().sort_values(ascending=False) * 100).round(2)
null_rate.head(25)

Time             0.0
AirTemp          0.0
Humidity         0.0
Pressure         0.0
Rainfall         0.0
TrackTemp        0.0
WindDirection    0.0
WindSpeed        0.0
season           0.0
event_round      0.0
session_type     0.0
ingestion_ts     0.0
dtype: float64

In [12]:
dfm = pd.read_parquet(paths["rmid"], engine="pyarrow")
dfl = pd.read_parquet(paths["rlast"], engine="pyarrow")

print("Round 1:", df1.shape)
print("Round mid:", dfm.shape)
print("Round last:", dfl.shape)

Round 1: (157, 12)
Round mid: (153, 12)
Round last: (143, 12)


In [13]:
cols1 = set(df1.columns)
colsm = set(dfm.columns)
colsl = set(dfl.columns)

print("Cols in r1 not in rmid:", sorted(cols1 - colsm)[:50])
print("Cols in rmid not in r1:", sorted(colsm - cols1)[:50])

print("Cols in r1 not in rlast:", sorted(cols1 - colsl)[:50])
print("Cols in rlast not in r1:", sorted(colsl - cols1)[:50])


Cols in r1 not in rmid: []
Cols in rmid not in r1: []
Cols in r1 not in rlast: []
Cols in rlast not in r1: []


In [14]:
shared = sorted(list(cols1 & colsm & colsl))

dtype_compare = pd.DataFrame({
    "round1":  df1[shared].dtypes.astype(str),
    "roundmid": dfm[shared].dtypes.astype(str),
    "roundlast": dfl[shared].dtypes.astype(str),
})

mismatched = dtype_compare[
    (dtype_compare["round1"] != dtype_compare["roundmid"]) |
    (dtype_compare["round1"] != dtype_compare["roundlast"])
]

mismatched.head(50)


,round1,roundmid,roundlast
